In [18]:
def proportion_waterbodies(fine_array, dem_info, coarse_da):
    coarse_shape = np.squeeze(coarse_da.values).shape
    coarse_array = np.full(coarse_shape, np.nan, dtype=np.float32)

    fine = fine_array.astype(np.float32).copy()

    reproject(source=fine,destination=coarse_array,src_transform=dem_info["transform"],src_crs=dem_info["crs"],
              dst_transform=coarse_da.rio.transform(),dst_crs=coarse_da.rio.crs,src_nodata=np.nan,dst_nodata=np.nan,
              resampling=Resampling.average)

    return coarse_array

def proportion_urban(fine_array, dem_info, coarse_da):
    coarse_shape = np.squeeze(coarse_da.values).shape
    coarse_array = np.full(coarse_shape, np.nan, dtype=np.float32)

    # 1 = urban/sub-urban, 0 = everything else
    urban = np.isin(fine_array, [20, 21]).astype(np.float32)

    # Preserve NaNs outside the catchment
    urban[np.isnan(fine_array)] = np.nan

    reproject(source=urban,destination=coarse_array,src_transform=dem_info["transform"],src_crs=dem_info["crs"],
              dst_transform=coarse_da.rio.transform(),dst_crs=coarse_da.rio.crs,src_nodata=np.nan,dst_nodata=np.nan,
              resampling=Resampling.average)

    return coarse_array


# Function to calculate proportion of each 5 km cell inside the catchment
def proportion_inside_catchment(inside_mask, src_transform, src_crs, coarse_da):
    coarse_shape = np.squeeze(coarse_da.values).shape

    # Count of 30 m cells inside each 5 km cell
    inside_count = np.full(coarse_shape, np.nan, dtype=np.float32)

    # Count of all 30 m cells contributing to each 5 km cell
    total_count = np.full(coarse_shape, np.nan, dtype=np.float32)

    # Reproject inside/outside mask
    reproject(source=inside_mask,destination=inside_count,src_transform=src_transform,src_crs=src_crs,
              dst_transform=coarse_da.rio.transform(),dst_crs=coarse_da.rio.crs,src_nodata=None,dst_nodata=np.nan,
              resampling=Resampling.sum)

    # Create an array of ones to count all contributing 30 m cells
    all_cells = np.ones_like(inside_mask, dtype=np.float32)

    reproject(source=all_cells,destination=total_count,src_transform=src_transform,src_crs=src_crs,
              dst_transform=coarse_da.rio.transform(),dst_crs=coarse_da.rio.crs,src_nodata=None,dst_nodata=np.nan,
              resampling=Resampling.sum)

    # Proportion inside catchment
    proportion = inside_count / total_count * 100

    # Completely outside catchment = 0 inside cells → NaN
    proportion[inside_count == 0] = np.nan

    return proportion

def compute_flow_unit_vectors(dem_da, dz_dx, dz_dy):
    y_coords = dem_da.y.values
    y_increases_with_row = y_coords[1] > y_coords[0] if len(y_coords) > 1 else True
    fx = -dz_dx
    fy = -dz_dy if y_increases_with_row else dz_dy
    mag = np.sqrt(fx**2 + fy**2)
    mag = np.where(mag == 0, np.nan, mag)
    return fx / mag, fy / mag


def edge_outward_scores(fx_cell, fy_cell, edge_width=3):
    ny, nx = fx_cell.shape
    ew = max(1, min(edge_width, ny // 2, nx // 2))
    if ny < 2 or nx < 2:
        return ({'N': np.nan, 'S': np.nan, 'E': np.nan, 'W': np.nan},
                {'N': True, 'S': True, 'E': True, 'W': True})

    north_fy, south_fy = fy_cell[:ew, :], fy_cell[-ew:, :]
    west_fx, east_fx = fx_cell[:, :ew], fx_cell[:, -ew:]

    truncated = {'N': np.all(np.isnan(north_fy)), 'S': np.all(np.isnan(south_fy)),
        'E': np.all(np.isnan(east_fx)), 'W': np.all(np.isnan(west_fx)),}
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', message='Mean of empty slice')
        scores = {'N': np.nanmean(north_fy), 'S': np.nanmean(-south_fy),
            'E': np.nanmean(east_fx), 'W': np.nanmean(-west_fx),}
    return scores, truncated

In [21]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import rioxarray as rxr
import warnings
import rasterio
from rasterio.mask import mask
from rasterio.warp import Resampling, reproject
from rasterio.features import geometry_mask
from shapely import box
import xarray as xr
from tqdm import tqdm

sys.path.append('../ProcessEvents/')
sys.path.append('../../ProcessCatchments')

from functions import filter_closer_to_catchment_xr_bounds
from helper_functions import extract_ha_num, find_matching_netcdf
# from functions_stage2 import (get_rainfall_cube_subsection, setup_worker_logger, find_max_precip_location_new, 
#                         find_temporal_profile_new, maybe_diagnose, analyse_peak_event,
#                              plot_cluster_check)

from config import RAINFALL_CSV_DIR, CATCHMENT_LOOKUP_DICT, OUT_DIR, CATCHMENTS, ENSEMBLE_MEMBERS 
NETCDF_DIR = "/scratch/hydro4/shared_data/climate_projections/UKCP18/UKCP_local/Soil_moisture/5km_regridded/"

event_details_fp = 'EventDetails'
rainfall_events_all_df = pd.read_pickle(f"/scratch/hydro4/users/kv25483/FutureFlood/Data/{event_details_fp}/all_catchments_added_soilvars.pkl")#
rainfall_events_all_df = rainfall_events_all_df[rainfall_events_all_df['max_precip']<130].copy()

In [26]:
cols = ["x_idx_global", "y_idx_global"]
rainfall_events_all_df["peak_cell_id"] = rainfall_events_all_df.groupby(cols, sort=False).ngroup()

In [28]:
subset = rainfall_events_all_df[:1000]

In [ ]:
# --- Output containers ---
peak_cell_id_order = []
peak_cell_slope_avg, peak_cell_slope_max = [], []
peak_cell_elev_avg, peak_cell_elev_max, peak_cell_elev_min = [], [], []
catchment_slope_avg, catchment_slope_max = [], []
catchment_elev_avg, catchment_elev_max, catchment_elev_min = [], [], []
peak_cell_sink_frac = []
peak_cell_outflow_N, peak_cell_outflow_S = [], []
peak_cell_outflow_E, peak_cell_outflow_W = [], []
peak_cell_edge_truncated_N, peak_cell_edge_truncated_S = [], []
peak_cell_edge_truncated_E, peak_cell_edge_truncated_W = [], []
peak_cell_near_catchment_boundary = []
peak_cell_water_prop, peak_cell_urban_prop, peak_cell_inside_prop = [], [], []

cellsize = 5000  # confirm this
edge_width_px = 3

for TARGET_HA in tqdm(np.unique(subset['catchment_num'])):
    boundary_gdf = CATCHMENTS[CATCHMENTS['HA_NUM'] == str(TARGET_HA)]
    CATCHMENT_POLY = boundary_gdf.geometry.iloc[0]

    # ==================================================
    # DEM / slope / sink / flow direction (once per catchment)
    # ==================================================
    dem_path = f"/scratch/hydro4/users/la17355/FUTURE-FLOOD/Data/Model_builds/Pluvial/v4/dem/dem_filled_30m_{TARGET_HA}.tif"
    sink_path = f"/scratch/hydro4/users/la17355/FUTURE-FLOOD/Data/Model_builds/Pluvial/v4/pluvial_sink_mask/pluvial_sink_mask_30m_{TARGET_HA}.tif"

    dem_da = rxr.open_rasterio(dem_path, masked=True).squeeze()
    sink_da = rxr.open_rasterio(sink_path, masked=True).squeeze()

    res_x, res_y = dem_da.rio.resolution()
    dz_dy, dz_dx = np.gradient(dem_da.values, abs(res_y), abs(res_x))
    slope_da = dem_da.copy(data=np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))))
    fx_vals, fy_vals = compute_flow_unit_vectors(dem_da, dz_dx, dz_dy)
    fx_da = dem_da.copy(data=fx_vals)
    fy_da = dem_da.copy(data=fy_vals)

    slope_catchment = slope_da.rio.clip([CATCHMENT_POLY], drop=True)
    dem_catchment = dem_da.rio.clip([CATCHMENT_POLY], drop=True)

    # ==================================================
    # Waterbodies / urban / inside-catchment proportion (5km grid, once per catchment)
    # ==================================================
    with rasterio.open(dem_path) as dem:
        dem_info = {"bounds": box(*dem.bounds), "transform": dem.transform, "crs": dem.crs,
                    "width": dem.width, "height": dem.height, "mask": dem.read(1)}

    soil_moisture_5km_fp = find_matching_netcdf(os.path.join(NETCDF_DIR, 'Ens_01'), 1991, 1, 1)
    with xr.open_dataset(soil_moisture_5km_fp) as soil_moisture_5km:
        grav_clipped = filter_closer_to_catchment_xr_bounds(
            soil_moisture_5km.isel(time=0), CATCHMENT_POLY)["moisture_content_of_soil_layer"]
        grav_clipped = grav_clipped.rio.write_crs("EPSG:27700")

    waterbodies_path = f'/scratch/hydro5/users/la17355/FUTURE-FLOOD/Results/Pluvial/associate_layers/waterbodies/waterbodies_{TARGET_HA}.tif'
    with rasterio.open(waterbodies_path) as src:
        waterbodies_data = src.read(1)
        valid = waterbodies_data != src.nodata
        waterbodies_data[~valid] = np.nan
    waterbodies_5km = proportion_waterbodies(waterbodies_data, dem_info, grav_clipped)

    landcover_path = "/scratch/hydro4/users/la17355/FUTURE-FLOOD/Data/CEH_landcover/resampled_30m/landcover_resampled_30m_nearest.tif"
    with rasterio.open(landcover_path) as src:
        landcover_data, _ = mask(src, [CATCHMENT_POLY], crop=True, nodata=0)
    landcover_data = landcover_data.astype(float)
    landcover_data[landcover_data == 0] = np.nan
    landcover_data = landcover_data[0]
    urban_5km = proportion_urban(landcover_data, dem_info, grav_clipped)

    with rasterio.open(landcover_path) as src:
        inside_mask = geometry_mask([CATCHMENT_POLY], out_shape=(src.height, src.width),
                                     transform=src.transform, invert=True)
    inside_mask = inside_mask.astype(np.float32)
    inside_proportion_5km = proportion_inside_catchment(inside_mask, src.transform, src.crs, grav_clipped)

    # ==================================================
    # Per-peak-cell loop, within this catchment
    # ==================================================
    this_catchment_events = subset[subset['catchment_num'] == TARGET_HA]

    for peak_cell_id in np.unique(this_catchment_events['peak_cell_id']):
        one_cell = this_catchment_events[this_catchment_events['peak_cell_id'] == peak_cell_id]
        x_coord = np.unique(one_cell['x_coord'])[0]
        y_coord = np.unique(one_cell['y_coord'])[0]
        y_idx = np.unique(one_cell['y_idx'])[0]   # local index into grav_clipped grid — confirm this
        x_idx = np.unique(one_cell['x_idx'])[0]   # local index into grav_clipped grid — confirm this

        cell_xmin, cell_xmax = x_coord - cellsize/2, x_coord + cellsize/2
        cell_ymin, cell_ymax = y_coord - cellsize/2, y_coord + cellsize/2

        slope_cell = slope_da.rio.clip_box(cell_xmin, cell_ymin, cell_xmax, cell_ymax)
        sink_cell = sink_da.rio.clip_box(cell_xmin, cell_ymin, cell_xmax, cell_ymax)
        dem_cell = dem_da.rio.clip_box(cell_xmin, cell_ymin, cell_xmax, cell_ymax)
        fx_cell = fx_da.rio.clip_box(cell_xmin, cell_ymin, cell_xmax, cell_ymax)
        fy_cell = fy_da.rio.clip_box(cell_xmin, cell_ymin, cell_xmax, cell_ymax)

        slope_vals, sink_vals, dem_vals = slope_cell.values, sink_cell.values, dem_cell.values

        if np.all(np.isnan(slope_vals)):
            mean_slope, max_slope = np.nan, np.nan
        else:
            mean_slope, max_slope = np.nanmean(slope_vals), np.nanmax(slope_vals)

        if np.all(np.isnan(dem_vals)):
            mean_elev, max_elev, min_elev = np.nan, np.nan, np.nan
        else:
            mean_elev, max_elev, min_elev = np.nanmean(dem_vals), np.nanmax(dem_vals), np.nanmin(dem_vals)

        sink_frac = np.nan if np.all(np.isnan(sink_vals)) else np.nanmean(sink_vals > 0)
        edge_scores, edge_truncated = edge_outward_scores(fx_cell.values, fy_cell.values, edge_width=edge_width_px)

        # --- water/urban/inside proportion via direct index lookup ---
        water_prop = waterbodies_5km[y_idx, x_idx]
        urban_prop = urban_5km[y_idx, x_idx]
        inside_prop = inside_proportion_5km[y_idx, x_idx]

        # --- append ---
        peak_cell_id_order.append(peak_cell_id)
        peak_cell_slope_avg.append(mean_slope); peak_cell_slope_max.append(max_slope)
        peak_cell_elev_avg.append(mean_elev); peak_cell_elev_max.append(max_elev); peak_cell_elev_min.append(min_elev)
        catchment_slope_avg.append(float(slope_catchment.mean())); catchment_slope_max.append(float(slope_catchment.max()))
        catchment_elev_avg.append(float(dem_catchment.mean())); catchment_elev_max.append(float(dem_catchment.max())); catchment_elev_min.append(float(dem_catchment.min()))
        peak_cell_sink_frac.append(sink_frac)
        peak_cell_outflow_N.append(edge_scores['N']); peak_cell_outflow_S.append(edge_scores['S'])
        peak_cell_outflow_E.append(edge_scores['E']); peak_cell_outflow_W.append(edge_scores['W'])
        peak_cell_edge_truncated_N.append(edge_truncated['N']); peak_cell_edge_truncated_S.append(edge_truncated['S'])
        peak_cell_edge_truncated_E.append(edge_truncated['E']); peak_cell_edge_truncated_W.append(edge_truncated['W'])
        peak_cell_near_catchment_boundary.append(any(edge_truncated.values()))
        peak_cell_water_prop.append(water_prop)
        peak_cell_urban_prop.append(urban_prop)
        peak_cell_inside_prop.append(inside_prop)

results_df = pd.DataFrame({
    'peak_cell_id': peak_cell_id_order,
    'peak_cell_slope_avg': peak_cell_slope_avg, 'peak_cell_slope_max': peak_cell_slope_max,
    'peak_cell_elev_avg': peak_cell_elev_avg, 'peak_cell_elev_max': peak_cell_elev_max, 'peak_cell_elev_min': peak_cell_elev_min,
    'catchment_slope_avg': catchment_slope_avg, 'catchment_slope_max': catchment_slope_max,
    'catchment_elev_avg': catchment_elev_avg, 'catchment_elev_max': catchment_elev_max, 'catchment_elev_min': catchment_elev_min,
    'peak_cell_sink_frac': peak_cell_sink_frac,
    'peak_cell_outflow_N': peak_cell_outflow_N, 'peak_cell_outflow_S': peak_cell_outflow_S,
    'peak_cell_outflow_E': peak_cell_outflow_E, 'peak_cell_outflow_W': peak_cell_outflow_W,
    'peak_cell_edge_truncated_N': peak_cell_edge_truncated_N, 'peak_cell_edge_truncated_S': peak_cell_edge_truncated_S,
    'peak_cell_edge_truncated_E': peak_cell_edge_truncated_E, 'peak_cell_edge_truncated_W': peak_cell_edge_truncated_W,
    'peak_cell_near_catchment_boundary': peak_cell_near_catchment_boundary,
    'peak_cell_water_prop': peak_cell_water_prop,
    'peak_cell_urban_prop': peak_cell_urban_prop,
    'peak_cell_inside_prop': peak_cell_inside_prop,})

subset = subset.merge(results_df, on='peak_cell_id', how='left')

 50%|███████████████████████████████████████████████████████████                                                           | 1/2 [02:17<02:17, 137.76s/it]

In [ ]:
subset.to_pickle(f"/scratch/hydro4/users/kv25483/FutureFlood/Data/{event_details_fp}/all_catchments.pkl")

### Sense checking with plots

In [9]:
TARGET_HA='23'

# ------------------------------------------------------
# Get catchment boundary
# ------------------------------------------------------
boundary_gdf    = CATCHMENTS[CATCHMENTS['HA_NUM'] == str(TARGET_HA)]
CATCHMENT_POLY = boundary_gdf.geometry.iloc[0]

# ------------------------------------------------------
# Get DEM files (for metadata to use in aggregation)
# ------------------------------------------------------
dem_files = [p for p in glob.glob(os.path.join(DEM_DIR, "*.tif"))
    if extract_ha_num(p) in [TARGET_HA]]
if not dem_files:
    raise FileNotFoundError(f"No DEM found for HA_NUM {TARGET_HA} in {DEM_DIR}")

dem_path = dem_files[0]
# ha_num = extract_ha_num(dem_path)

with rasterio.open(dem_path) as dem:
    dem_info = {"bounds": box(*dem.bounds),"transform": dem.transform,"crs": dem.crs,"width": dem.width,
                "height": dem.height,"mask": dem.read(1), }

# ------------------------------------------------------
# Get 5 km soil moisture data (for using to define 5km grid to aggregate to) 
# ------------------------------------------------------    
soil_moisture_5km_fp = find_matching_netcdf(os.path.join(NETCDF_DIR, 'Ens_01'), 1991, 1,1 )
with xr.open_dataset(soil_moisture_5km_fp) as soil_moisture_5km:
    grav_clipped = filter_closer_to_catchment_xr_bounds(
            soil_moisture_5km.isel(time=0), CATCHMENT_POLY)["moisture_content_of_soil_layer"]
    grav_clipped = grav_clipped.rio.write_crs("EPSG:27700")    
    
    
# ------------------------------------------------------
# Get waterbodies data and aggregate to 5km (proportion of cells which are water)
# ------------------------------------------------------    
waterbodies_path = f'/scratch/hydro5/users/la17355/FUTURE-FLOOD/Results/Pluvial/associate_layers/waterbodies/waterbodies_{TARGET_HA}.tif'
with rasterio.open(waterbodies_path) as src:
    waterbodies_data = src.read(1)
    valid = waterbodies_data != src.nodata
    waterbodies_data[~valid] = np.nan
    
waterbodies_5km = proportion_waterbodies(waterbodies_data, dem_info, grav_clipped)

# ------------------------------------------------------
# Get landcover data and aggregate to 5km (proportion of cells which are urban/semi-urban)
# ------------------------------------------------------     
landcover_path = "/scratch/hydro4/users/la17355/FUTURE-FLOOD/Data/CEH_landcover/resampled_30m/landcover_resampled_30m_nearest.tif"
with rasterio.open(landcover_path) as src:
    landcover_data, landcover_transform = mask(src,[CATCHMENT_POLY],crop=True,nodata=0)

landcover_data = landcover_data.astype(float)
landcover_data[landcover_data == 0] = np.nan
landcover_data = landcover_data[0]

urban_5km = proportion_urban(landcover_data, dem_info, grav_clipped)  

# ------------------------------------------------------
# Find the proportion of 30m cells outside the catchment boundary within each 5km cell
# ------------------------------------------------------ 
# Create a 30 m mask: 1 = inside catchment, 0 = outside
with rasterio.open(landcover_path) as src:
    inside_mask = geometry_mask([CATCHMENT_POLY],out_shape=(src.height, src.width),transform=src.transform,invert=True)

inside_mask = inside_mask.astype(np.float32)
inside_proportion_5km = proportion_inside_catchment(inside_mask,src.transform,src.crs,grav_clipped)
inside_da = grav_clipped.copy(data=inside_proportion_5km)

fig, ax = plt.subplots(figsize=(8, 8))
inside_da.plot(ax=ax, cmap="viridis",
               cbar_kwargs={"label": "Number of 30 m cells inside catchment"})
boundary_gdf.boundary.plot(ax=ax, edgecolor="red", linewidth=2, label="Catchment boundary")
ax.set_aspect("equal")
ax.legend()
plt.show()



waterbodies_5km_da = grav_clipped.copy(data=waterbodies_5km)
fig, ax = plt.subplots(figsize=(8, 8))
waterbodies_5km_da.plot(ax=ax, cmap="viridis",
               cbar_kwargs={"label": "Proportion of waterbody cells inside catchment"})
boundary_gdf.boundary.plot(ax=ax, edgecolor="red", linewidth=2, label="Catchment boundary")
ax.set_aspect("equal")
ax.legend()
plt.show()

urban_5km_da = grav_clipped.copy(data=waterbodies_5km)

fig, ax = plt.subplots(figsize=(8, 8))
urban_5km_da.plot(ax=ax, cmap="viridis",
               cbar_kwargs={"label": "Proportion of urban cells inside catchment"})
boundary_gdf.boundary.plot(ax=ax, edgecolor="red", linewidth=2, label="Catchment boundary")
ax.set_aspect("equal")
ax.legend()
plt.show()

/tmp/ipykernel_2206616/4214438161.py:74: RuntimeWarning: Mean of empty slice
  'N': np.nanmean(north_fy),
/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:74: RuntimeWarning: Mean of empty slice
  'N': np.nanmean(north_fy),
/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:74: RuntimeWarning: Mean of empty slice
  'N': np.nanmean(north_fy),
/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:76: Run

/tmp/ipykernel_2206616/4214438161.py:74: RuntimeWarning: Mean of empty slice
  'N': np.nanmean(north_fy),
/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:74: RuntimeWarning: Mean of empty slice
  'N': np.nanmean(north_fy),
/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:75: Ru

/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:74: RuntimeWarning: Mean of empty slice
  'N': np.nanmean(north_fy),
/tmp/ipykernel_2206616/4214438161.py:74: RuntimeWarning: Mean of empty slice
  'N': np.nanmean(north_fy),
/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:76: RuntimeWarning: Mean of empty slice
  'E': np.nanmean(east_fx),
/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:75: Run

/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:74: RuntimeWarning: Mean of empty slice
  'N': np.nanmean(north_fy),
/tmp/ipykernel_2206616/4214438161.py:74: RuntimeWarning: Mean of empty slice
  'N': np.nanmean(north_fy),
/tmp/ipykernel_2206616/4214438161.py:76: RuntimeWarning: Mean of empty slice
  'E': np.nanmean(east_fx),
/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:76: RuntimeWarning: Mean of empty slice
  'E': np.nanmean(east_fx),
/tmp/ipykernel_2206616/4214438161.py:76: RuntimeWarning: Mean of empty slice
  'E': np.nanmean(east_fx),
/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:74: RuntimeWarning: Mean of empty slice
  'N': np.nanmean(north_fy),
/tmp/ipykernel_2206616/4214438161.py:75: Runtime

/tmp/ipykernel_2206616/4214438161.py:76: RuntimeWarning: Mean of empty slice
  'E': np.nanmean(east_fx),
/tmp/ipykernel_2206616/4214438161.py:76: RuntimeWarning: Mean of empty slice
  'E': np.nanmean(east_fx),
/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:76: RuntimeWarning: Mean of empty slice
  'E': np.nanmean(east_fx),
/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:74: RuntimeWarning: Mean of empty slice
  'N': np.nanmean(north_fy),
/tmp/ipykernel_2206616/4214438161.py:76: RuntimeWarning: Mean of empty slice
  'E': np.nanmean(east_fx),
/tmp/ipykernel_2206616/4214438161.py:77: Runtim

...

/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:74: RuntimeWarning: Mean of empty slice
  'N': np.nanmean(north_fy),
/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:74: RuntimeWarning: Mean of empty slice
  'N': np.nanmean(north_fy),
/tmp/ipykernel_2206616/4214438161.py:74: RuntimeWarning: Mean of empty slice
  'N': np.nanmean(north_fy),
/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:74: 

/tmp/ipykernel_2206616/4214438161.py:74: RuntimeWarning: Mean of empty slice
  'N': np.nanmean(north_fy),
/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:74: RuntimeWarning: Mean of empty slice
  'N': np.nanmean(north_fy),
/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:76: RuntimeWarning: Mean of empty slice
  'E': np.nanmean(east_fx),
/tmp/ipykernel_2206616/4214438161.py:76: RuntimeWarning: Mean of empty slice
  'E': np.nanmean(east_fx),
/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:77: Runti

/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:74: RuntimeWarning: Mean of empty slice
  'N': np.nanmean(north_fy),
/tmp/ipykernel_2206616/4214438161.py:76: RuntimeWarning: Mean of empty slice
  'E': np.nanmean(east_fx),
/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:77: Ru

/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:74: RuntimeWarning: Mean of empty slice
  'N': np.nanmean(north_fy),
/tmp/ipykernel_2206616/4214438161.py:74: RuntimeWarning: Mean of empty slice
  'N': np.nanmean(north_fy),
/tmp/ipykernel_2206616/4214438161.py:74: RuntimeWarning: Mean of empty slice
  'N': np.nanmean(north_fy),
/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:76: RuntimeWarning: Mean of empty slice
  'E': np.nanmean(east_fx),
/tmp/ipykernel_2206616/4214438161.py:76: RuntimeWarning: Mean of empty slice
  'E': np.nanmean(east_fx),
/tmp/ipykernel_2206616/4214438161.py:76: RuntimeWarning: Mean of empty slice
  'E': np.nanmean(east_fx),
/tmp/ipykernel_2206616/4214438161.py:74: Runtime

/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:74: RuntimeWarning: Mean of empty slice
  'N': np.nanmean(north_fy),
/tmp/ipykernel_2206616/4214438161.py:76: RuntimeWarning: Mean of empty slice
  'E': np.nanmean(east_fx),
/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:75: RuntimeWarning: Mean of empty slice
  'S': np.nanmean(-south_fy),
/tmp/ipykernel_2206616/4214438161.py:77: RuntimeWarning: Mean of empty slice
  'W': np.nanmean(-west_fx),
/tmp/ipykernel_2206616/4214438161.py:75: Run